# 📊 Paper baselines · Qwen3.6-27B entity-recognition

**Goal**: produce the missing reviewer-defensible numbers for the ICML 2026 Mech Interp Workshop submission. Notebook 24b gave us a single-latent SAE feature AUROC of **0.8379 at L11/f61723**. Reviewers will ask three things 24b did not answer:

1. **What does a linear probe on raw residuals achieve?** If a probe at L11 reaches 0.95, the SAE feature is *redundant*. If it reaches ~0.84, the SAE feature is *interpretably matching the supervised ceiling*. Ferrando 2024 reports this baseline; we did not.
2. **What does a diff-of-means probe achieve?** (Farquhar 2023's critique of CCS — always ship the simplest possible baseline.)
3. **Where exactly is the entity-recognition signal?** Notebook 24b only tested L11/L31/L55 because that's where the SAE lives. For the *novel finding* that L11 (early) is best, we need the per-layer scan across all 64 layers.

Plus 95% bootstrap CI on every AUROC (N≈226 is small, point estimates are fragile).

**Outputs**:
- `paper_baselines_per_layer.json` — AUROC + 95% CI for SAE feature / linear probe / diff-of-means × 64 layers
- `paper_baselines_per_type.json` — per-type breakdown at L11
- `fig_per_layer_comparison.pdf` — the headline figure
- Uploaded to `caiovicentino1/qwen36-27b-sae-papergrade/paper_baselines/`

**Compute**: ~3-4h on a single A100 80GB (or RTX 6000 Pro). 27B forward pass × 226 prompts dominates.

In [ ]:
!pip install -q -U transformers accelerate safetensors huggingface_hub datasets scikit-learn matplotlib tqdm

## 1. Config + load Qwen3.6-27B + 3 SAEs

In [ ]:
HF_SAE_REPO   = 'caiovicentino1/qwen36-27b-sae-papergrade'
HF_BASE_MODEL = 'Qwen/Qwen3.6-27B'
SAE_LAYERS    = [11, 31, 55]
ALL_LAYERS    = list(range(64))   # full per-layer scan
D_MODEL       = 5120
D_SAE         = 65_536
K             = 128

N_PER_TYPE    = 250
N_ATTRIBUTES_TO_TEST = 3
MAX_GEN_TOKENS = 24
N_PILE_TOKENS = 2000
PILE_FILTER_THRESHOLD = 0.02

TRAIN_FRAC = 0.7
BOOTSTRAP_N = 1000
BOOTSTRAP_SEED = 0
SEED = 0

BEST_FEATURE_PRIOR = 61723   # from notebook 24b — confirm survives the re-run
BEST_LAYER_PRIOR   = 11

import os
# expandable_segments avoids the OOM in transformers' caching_allocator_warmup
# on 27B models. MUST be set BEFORE `import torch`.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import math, json, time, random, re
import numpy as np
import torch
random.seed(SEED); torch.manual_seed(SEED); np.random.seed(SEED)

from huggingface_hub import login, hf_hub_download
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    login()

from transformers import AutoTokenizer, AutoModelForImageTextToText
from safetensors.torch import load_file
import torch.nn.functional as F

device = 'cuda'
tok = AutoTokenizer.from_pretrained(HF_BASE_MODEL, trust_remote_code=True)
# device_map='auto' + max_memory leaves headroom for activations + SAEs.
# 80% cap fits 54GB model on a 95GB GPU and avoids the 50GB warmup OOM.
total_vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
max_model_gib  = max(60, int(total_vram_gib * 0.80))
print(f'GPU total VRAM: {total_vram_gib:.1f} GiB · capping model at {max_model_gib} GiB')
model = AutoModelForImageTextToText.from_pretrained(
    HF_BASE_MODEL,
    dtype=torch.bfloat16,
    attn_implementation='sdpa',
    device_map='auto',
    max_memory={0: f'{max_model_gib}GiB', 'cpu': '40GiB'},
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

class TopKSAE(torch.nn.Module):
    def __init__(self, sd, k):
        super().__init__()
        self.W_enc = torch.nn.Parameter(sd['W_enc'].to(torch.bfloat16), requires_grad=False)
        self.b_enc = torch.nn.Parameter(sd['b_enc'].to(torch.bfloat16), requires_grad=False)
        self.W_dec = torch.nn.Parameter(sd['W_dec'].to(torch.bfloat16), requires_grad=False)
        self.b_dec = torch.nn.Parameter(sd['b_dec'].to(torch.bfloat16), requires_grad=False)
        self.k = k
    def encode(self, x):
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc
        vals, idx = pre.topk(self.k, dim=-1)
        vals = F.relu(vals)
        z = torch.zeros_like(pre)
        z.scatter_(-1, idx, vals)
        return z

saes = {}
for layer in SAE_LAYERS:
    path = hf_hub_download(HF_SAE_REPO, f'sae_L{layer}_latest.safetensors')
    saes[layer] = TopKSAE(load_file(path), K).to(device).eval()
    print(f'  ✓ SAE L{layer}')

n_layers = len(model.model.language_model.layers)
print(f'\nmodel has {n_layers} layers · vram free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')
assert n_layers == 64, f'expected 64 layers, got {n_layers}'

## 2. Pull Ferrando's pre-processed entity files (same as 24b)

In [ ]:
import requests
from pathlib import Path

# Schema: each file is a list of {entity, entity_type, attributes:[{attribute_type, attribute_value}, ...]}
FERRANDO_RAW = 'https://raw.githubusercontent.com/javiferran/sae_entities/main/dataset/processed/entities'
ENTITY_TYPES = ['player', 'movie', 'city', 'song']

def get_attr(entry, attr_type):
    """Return attribute_value (str) or None if missing."""
    for a in entry.get('attributes', []):
        if a.get('attribute_type') == attr_type:
            v = a.get('attribute_value')
            if isinstance(v, list):
                return ' '.join(str(x) for x in v)
            return str(v) if v is not None else None
    return None

entities_by_type = {}
for et in ENTITY_TYPES:
    url = f'{FERRANDO_RAW}/{et}.json'
    r = requests.get(url)
    r.raise_for_status()
    data = r.json()
    random.Random(SEED).shuffle(data)
    entities_by_type[et] = data[:N_PER_TYPE]
    print(f'  {et}: {len(data)} → sampled {len(entities_by_type[et])}')
    sample = entities_by_type[et][0]
    attr_types = [a['attribute_type'] for a in sample.get('attributes', [])]
    print(f'    sample entity: {sample.get("entity")}')
    print(f'    attribute_types: {attr_types}')

## 3. Attribute-recall labelling (deterministic, same as 24b)

In [ ]:
from tqdm.auto import tqdm

REFUSAL_PATTERNS = [
    r"i (do not|don't) (know|have|recognize|recognise)",
    r"i'?m not (sure|familiar|aware)",
    r"no (information|data|record)",
    r"(unable|cannot|can't) (find|locate|provide)",
    r"not (familiar|aware) with",
    r"(unknown|unclear) to me",
    r"i (do not|don't) (recall|recollect)",
]
REFUSAL_RE = re.compile('|'.join(REFUSAL_PATTERNS), re.IGNORECASE)

def is_refusal(text: str) -> bool:
    return REFUSAL_RE.search(text or '') is not None

def chat_query(q: str, max_new=MAX_GEN_TOKENS) -> str:
    msgs = [{'role': 'user', 'content': q}]
    ids = tok.apply_chat_template(
        msgs, return_tensors='pt', add_generation_prompt=True,
        enable_thinking=False,
    ).to(device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    text = tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)
    return text.strip()

# Templates aligned with the actual Ferrando schema (attribute_type names from `dataset/processed/entities/{type}.json`).
ATTRIBUTE_TEMPLATES = {
    'player': [
        ('place_birth', "In which city or country was the football/basketball player {entity} born?"),
        ('teams_list',  "Name one professional team that player {entity} has played for."),
        ('height',      "Approximately how tall is athlete {entity} (in cm)?"),
    ],
    'movie': [
        ('directors',   "Who directed the movie {entity}?"),
        ('release_year',"In what year was the movie {entity} released?"),
        ('cast',        "Name one main actor in the movie {entity}."),
    ],
    'city': [
        ('country',     "In which country is the city {entity} located?"),
        ('location',    "What region or area of its country is the city {entity} in?"),
        ('population',  "Roughly what is the population of the city {entity}?"),
    ],
    'song': [
        ('performers',  "Who is the artist or band that performs the song {entity}?"),
        ('publication_year', "In what year was the song {entity} released?"),
        ('genres',      "What is the musical genre of the song {entity}?"),
    ],
}

def label_entity(entry, etype):
    name = entry['entity']
    correct = 0; refused = 0; answers = []
    for attr_key, template in ATTRIBUTE_TEMPLATES[etype][:N_ATTRIBUTES_TO_TEST]:
        ans = chat_query(template.format(entity=name))
        answers.append({'attr': attr_key, 'q': template, 'a': ans})
        if is_refusal(ans):
            refused += 1
            continue
        gold_str = get_attr(entry, attr_key) or ''
        # token-overlap heuristic: any non-trivial gold token appears in answer (case-insensitive)
        gold_tokens = [t for t in re.findall(r'\w+', gold_str.lower()) if len(t) >= 3]
        ans_low = ans.lower()
        hit = any(t in ans_low for t in gold_tokens)
        if hit:
            correct += 1
    if correct >= 2:
        cls = 'known'
    elif correct == 0 and refused >= 1:
        cls = 'unknown'
    else:
        cls = 'middle'
    return {'entity': name, 'type': etype, 'class': cls,
            'correct': correct, 'refused': refused, 'answers': answers}

labelled = []
for etype, entries in entities_by_type.items():
    for entry in tqdm(entries, desc=f'label·{etype}'):
        labelled.append(label_entity(entry, etype))

from collections import Counter
ctr = Counter((l['class'], l['type']) for l in labelled)
print('\nlabel distribution:')
for k in sorted(ctr.keys()):
    print(f'  {k}: {ctr[k]}')

n_known   = sum(1 for l in labelled if l['class'] == 'known')
n_unknown = sum(1 for l in labelled if l['class'] == 'unknown')
print(f'\ntotal known={n_known}, unknown={n_unknown}')

## 4. Capture residuals at ALL 64 layers + SAE-encoded vectors at L11/L31/L55

One forward pass per entity. We use `output_hidden_states=True` to get all 65 layer outputs (embedding + 64 transformer blocks) in a single shot. At the SAE-trained layers we additionally encode through the SAE.

Position: last token of the entity name inside the prompt (same as 24b).

In [ ]:
PROMPT_TEMPLATE = "What can you tell me about '{entity}'?"

def find_entity_last_pos(prompt: str) -> int:
    full_ids = tok(prompt, return_tensors='pt')['input_ids'][0].tolist()
    close_q = tok.encode("'?", add_special_tokens=False)
    n = len(full_ids); m = len(close_q)
    for i in range(n - m, -1, -1):
        if full_ids[i:i+m] == close_q:
            return i - 1
    return n - 2

labelled_kept = [l for l in labelled if l['class'] in ('known', 'unknown')]
y_label = np.array([1 if l['class'] == 'known' else 0 for l in labelled_kept])
type_label = np.array([l['type'] for l in labelled_kept])
n_total = len(labelled_kept)
print(f'kept {n_total} entries (known + unknown only)')

# Pre-allocate residual cache: (N, 64, D_MODEL) in float32 on CPU
residuals_all = np.zeros((n_total, 64, D_MODEL), dtype=np.float32)
z_at_sae = {layer: np.zeros((n_total, D_SAE), dtype=np.float32) for layer in SAE_LAYERS}

with torch.no_grad():
    for i, entry in enumerate(tqdm(labelled_kept, desc='forward·all-layers')):
        prompt = PROMPT_TEMPLATE.format(entity=entry['entity'])
        ids = tok(prompt, return_tensors='pt')['input_ids'].to(device)
        pos = find_entity_last_pos(prompt)
        out = model(ids, output_hidden_states=True)
        # out.hidden_states is a tuple of length n_layers+1 (embedding + each transformer block)
        hs = out.hidden_states  # ((1, T, D), ...)
        for L in range(64):
            # block L's output is hidden_states[L+1] (index 0 is embedding)
            residuals_all[i, L] = hs[L + 1][0, pos].float().cpu().numpy()
        for L in SAE_LAYERS:
            r = torch.tensor(residuals_all[i, L], dtype=torch.bfloat16, device=device)
            z_at_sae[L][i] = saes[L].encode(r.unsqueeze(0))[0].float().cpu().numpy()
        del out
        if i % 25 == 0:
            torch.cuda.empty_cache()

print(f'\nresiduals_all: {residuals_all.shape} ({residuals_all.nbytes/1e6:.1f} MB)')
for L in SAE_LAYERS:
    print(f'  z_at_sae[L{L}]: {z_at_sae[L].shape}')
print(f'class balance: known={int(y_label.sum())}, unknown={int(n_total-y_label.sum())}')

## 5. Bootstrap helpers

1000 bootstrap resamples on the test set. Returns (mean, lo, hi) at 95% CI.

In [ ]:
from sklearn.metrics import roc_auc_score

def bootstrap_auroc(y_true, scores, n_boot=BOOTSTRAP_N, seed=BOOTSTRAP_SEED):
    """Stratified-by-default bootstrap. Returns (mean, lo, hi) at 95% CI."""
    rng = np.random.default_rng(seed)
    n = len(y_true)
    aurocs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        if len(np.unique(y_true[idx])) < 2:
            continue
        try:
            aurocs.append(roc_auc_score(y_true[idx], scores[idx]))
        except ValueError:
            continue
    aurocs = np.array(aurocs)
    if aurocs.size == 0:
        return float('nan'), float('nan'), float('nan')
    return float(aurocs.mean()), float(np.percentile(aurocs, 2.5)), float(np.percentile(aurocs, 97.5))

# Same train/test split as 24b
np.random.seed(SEED)
all_idx = np.arange(n_total)
np.random.shuffle(all_idx)
n_train = int(TRAIN_FRAC * n_total)
tr_idx, te_idx = all_idx[:n_train], all_idx[n_train:]
y_tr, y_te = y_label[tr_idx], y_label[te_idx]
print(f'train: {len(tr_idx)} (k={int(y_tr.sum())}, u={int(len(y_tr)-y_tr.sum())})')
print(f'test:  {len(te_idx)} (k={int(y_te.sum())}, u={int(len(y_te)-y_te.sum())})')

## 6. Method 1 — Single-latent SAE feature AUROC + bootstrap CI

At L11/L31/L55: rank features by Cohen's d on TRAIN, apply Pile filter, evaluate top-100 single-latent AUROCs on TEST. Report best per layer with 95% CI.

In [ ]:
# Pile filter: drop features active on >2% of Pile tokens (same as 24b)
from datasets import load_dataset

fire_rate = {}
pile_iter = load_dataset('NeelNanda/pile-10k', split='train', streaming=True)
pile_texts = []
for ex in pile_iter:
    pile_texts.append(ex['text'])
    if len(pile_texts) >= 50: break  # ~2000 tokens worth

fire_count = {L: np.zeros(D_SAE, dtype=np.int64) for L in SAE_LAYERS}
n_pile_tok_total = 0
with torch.no_grad():
    for txt in tqdm(pile_texts, desc='pile-fire-rate'):
        ids = tok(txt, return_tensors='pt', truncation=True, max_length=512)['input_ids'].to(device)
        out = model(ids, output_hidden_states=True)
        for L in SAE_LAYERS:
            r = out.hidden_states[L + 1][0].to(torch.bfloat16)
            z = saes[L].encode(r).float().cpu().numpy()
            fire_count[L] += (z > 0).sum(axis=0)
        n_pile_tok_total += ids.shape[1]
        del out
for L in SAE_LAYERS:
    fire_rate[L] = fire_count[L] / n_pile_tok_total
    n_noisy = int((fire_rate[L] > PILE_FILTER_THRESHOLD).sum())
    print(f'  L{L}: {n_noisy}/{D_SAE} features above {PILE_FILTER_THRESHOLD*100:.1f}% (will be filtered)')
print(f'pile tokens used: {n_pile_tok_total}')

In [ ]:
TOP_FEATURES_TO_TEST = 100

sae_results = {}
for L in SAE_LAYERS:
    Z = z_at_sae[L]
    Z_tr = Z[tr_idx]
    Zk_tr = Z_tr[y_tr == 1]; Zu_tr = Z_tr[y_tr == 0]
    mu_k, mu_u = Zk_tr.mean(0), Zu_tr.mean(0)
    sd = np.sqrt((Zk_tr.std(0)**2 + Zu_tr.std(0)**2) / 2) + 1e-9
    sep = (mu_k - mu_u) / sd
    pile_mask = fire_rate[L] <= PILE_FILTER_THRESHOLD
    sep_filtered = np.where(pile_mask, sep, 0.0)
    top_feats = np.argsort(-np.abs(sep_filtered))[:TOP_FEATURES_TO_TEST]

    best = {'feat': None, 'auroc_pt': -1.0}
    Z_te = Z[te_idx]
    for feat in top_feats:
        score = Z_te[:, feat]
        try:
            auc = roc_auc_score(y_te, score)
        except ValueError:
            continue
        if auc < 0.5: auc = 1 - auc; score = -score
        if auc > best['auroc_pt']:
            best = {'feat': int(feat), 'auroc_pt': float(auc), 'score': score, 'flipped': sep_filtered[feat] < 0}
    if best['feat'] is None:
        sae_results[L] = None
        continue
    mean, lo, hi = bootstrap_auroc(y_te, best['score'])
    sae_results[L] = {
        'best_feature':  best['feat'],
        'auroc_point':   best['auroc_pt'],
        'auroc_mean':    mean,
        'auroc_ci_lo':   lo,
        'auroc_ci_hi':   hi,
        'sep_at_best':   float(sep_filtered[best['feat']]),
        'pile_rate':     float(fire_rate[L][best['feat']]),
    }
    print(f'  L{L}: f{best["feat"]} AUROC={best["auroc_pt"]:.4f}  CI=[{lo:.3f}, {hi:.3f}]  sep={sep_filtered[best["feat"]]:+.2f}')

## 7. Method 2 — Linear probe per layer (sklearn LogisticRegression)

L2-regularized logistic regression on raw residual streams. Fit on TRAIN, eval on TEST with 95% bootstrap CI. Run for **all 64 layers**.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

linprobe_results = {}
for L in tqdm(ALL_LAYERS, desc='linprobe'):
    X_tr = residuals_all[tr_idx, L]
    X_te = residuals_all[te_idx, L]
    sc = StandardScaler().fit(X_tr)
    X_tr_s = sc.transform(X_tr); X_te_s = sc.transform(X_te)
    clf = LogisticRegression(penalty='l2', C=1.0, max_iter=2000, random_state=SEED, solver='lbfgs')
    clf.fit(X_tr_s, y_tr)
    score_te = clf.decision_function(X_te_s)
    pt = roc_auc_score(y_te, score_te)
    mean, lo, hi = bootstrap_auroc(y_te, score_te)
    linprobe_results[L] = {
        'auroc_point':  float(pt),
        'auroc_mean':   mean,
        'auroc_ci_lo':  lo,
        'auroc_ci_hi':  hi,
    }
best_linprobe_layer = max(linprobe_results.keys(), key=lambda L: linprobe_results[L]['auroc_point'])
print(f'best linear probe layer: L{best_linprobe_layer}  AUROC={linprobe_results[best_linprobe_layer]["auroc_point"]:.4f}'
      f'  CI=[{linprobe_results[best_linprobe_layer]["auroc_ci_lo"]:.3f},'
      f' {linprobe_results[best_linprobe_layer]["auroc_ci_hi"]:.3f}]')
for L in [11, 31, 55]:
    r = linprobe_results[L]
    print(f'  L{L}: linprobe AUROC={r["auroc_point"]:.4f}  CI=[{r["auroc_ci_lo"]:.3f}, {r["auroc_ci_hi"]:.3f}]')

## 8. Method 3 — Diff-of-means probe per layer (Farquhar 2023 baseline)

Direction = unit vector of (mean of known − mean of unknown) on TRAIN. Test score = ⟨residual, direction⟩. Simplest possible probe; if it ties LR, the linear separability is along a single direction.

In [ ]:
diffmeans_results = {}
for L in tqdm(ALL_LAYERS, desc='diff-of-means'):
    X_tr = residuals_all[tr_idx, L]
    X_te = residuals_all[te_idx, L]
    mu_k = X_tr[y_tr == 1].mean(0)
    mu_u = X_tr[y_tr == 0].mean(0)
    direction = mu_k - mu_u
    direction /= (np.linalg.norm(direction) + 1e-12)
    score_te = X_te @ direction
    pt = roc_auc_score(y_te, score_te)
    if pt < 0.5:
        score_te = -score_te; pt = 1 - pt
    mean, lo, hi = bootstrap_auroc(y_te, score_te)
    diffmeans_results[L] = {
        'auroc_point':  float(pt),
        'auroc_mean':   mean,
        'auroc_ci_lo':  lo,
        'auroc_ci_hi':  hi,
    }
best_dm_layer = max(diffmeans_results.keys(), key=lambda L: diffmeans_results[L]['auroc_point'])
print(f'best diff-of-means layer: L{best_dm_layer}  AUROC={diffmeans_results[best_dm_layer]["auroc_point"]:.4f}')
for L in [11, 31, 55]:
    r = diffmeans_results[L]
    print(f'  L{L}: diff-of-means AUROC={r["auroc_point"]:.4f}  CI=[{r["auroc_ci_lo"]:.3f}, {r["auroc_ci_hi"]:.3f}]')

## 9. Headline figure — per-layer comparison

AUROC vs layer for the 3 methods. SAE feature is shown as 3 stars at L11/L31/L55; linear probe and diff-of-means are continuous lines across all 64 layers.

In [ ]:
import matplotlib.pyplot as plt

Ls = np.array(ALL_LAYERS)
lp_pt  = np.array([linprobe_results[L]['auroc_point'] for L in Ls])
lp_lo  = np.array([linprobe_results[L]['auroc_ci_lo'] for L in Ls])
lp_hi  = np.array([linprobe_results[L]['auroc_ci_hi'] for L in Ls])
dm_pt  = np.array([diffmeans_results[L]['auroc_point'] for L in Ls])
dm_lo  = np.array([diffmeans_results[L]['auroc_ci_lo'] for L in Ls])
dm_hi  = np.array([diffmeans_results[L]['auroc_ci_hi'] for L in Ls])

fig, ax = plt.subplots(figsize=(7.5, 4.0))
ax.plot(Ls, lp_pt, '-', color='#1f77b4', linewidth=1.6, label='Linear probe (L2 LR)')
ax.fill_between(Ls, lp_lo, lp_hi, color='#1f77b4', alpha=0.15)
ax.plot(Ls, dm_pt, '-', color='#7f7f7f', linewidth=1.4, label='Diff-of-means probe')
ax.fill_between(Ls, dm_lo, dm_hi, color='#7f7f7f', alpha=0.12)
for L in SAE_LAYERS:
    if sae_results.get(L):
        r = sae_results[L]
        ax.errorbar([L], [r['auroc_point']],
                    yerr=[[r['auroc_point']-r['auroc_ci_lo']], [r['auroc_ci_hi']-r['auroc_point']]],
                    fmt='*', color='#d62728', markersize=14, capsize=4,
                    label='Single SAE latent' if L == SAE_LAYERS[0] else None)
ax.axhline(0.732, linestyle='--', color='#666', linewidth=0.8)
ax.text(63, 0.738, 'Ferrando 2024 (Gemma-2-2B-IT, L13)', ha='right', va='bottom', fontsize=8, color='#666')
ax.set_xlabel('Layer index')
ax.set_ylabel('AUROC (known vs unknown · test split)')
ax.set_title('Entity-recognition signal across Qwen3.6-27B (N≈226)')
ax.set_ylim(0.45, 1.0)
ax.set_xlim(-1, 64)
ax.legend(loc='lower right', frameon=False)
ax.grid(alpha=0.3)
plt.tight_layout()
fig_path = '/content/fig_per_layer_comparison.pdf'
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.savefig(fig_path.replace('.pdf', '.png'), dpi=200, bbox_inches='tight')
plt.show()
print(f'saved → {fig_path}')

## 10. Per-type breakdown at L11 with bootstrap CI

For player / movie / city / song: SAE feature L11/f-best, linear probe L11, diff-of-means L11.

In [ ]:
type_breakdown = {}
L = 11
Z = z_at_sae[L]
best_sae_feat = sae_results[L]['best_feature']
X_tr_L = residuals_all[tr_idx, L]
X_te_L = residuals_all[te_idx, L]
sc = StandardScaler().fit(X_tr_L)
clf = LogisticRegression(penalty='l2', C=1.0, max_iter=2000, random_state=SEED).fit(sc.transform(X_tr_L), y_tr)
mu_k = X_tr_L[y_tr == 1].mean(0); mu_u = X_tr_L[y_tr == 0].mean(0)
dm_dir = (mu_k - mu_u) / (np.linalg.norm(mu_k - mu_u) + 1e-12)

for t in ENTITY_TYPES:
    mask_te = (type_label[te_idx] == t)
    if mask_te.sum() < 10 or len(np.unique(y_te[mask_te])) < 2:
        type_breakdown[t] = None; continue
    sae_score = Z[te_idx][mask_te, best_sae_feat]
    if sae_results[L].get('sep_at_best', 0) < 0:
        sae_score = -sae_score
    lp_score  = clf.decision_function(sc.transform(X_te_L[mask_te]))
    dm_score  = X_te_L[mask_te] @ dm_dir
    yt = y_te[mask_te]
    type_breakdown[t] = {
        'n_test':     int(mask_te.sum()),
        'n_known':    int(yt.sum()),
        'n_unknown':  int(len(yt) - yt.sum()),
        'sae_feature':  dict(zip(['mean','lo','hi'], bootstrap_auroc(yt, sae_score))),
        'linear_probe': dict(zip(['mean','lo','hi'], bootstrap_auroc(yt, lp_score))),
        'diff_means':   dict(zip(['mean','lo','hi'], bootstrap_auroc(yt, dm_score))),
    }
    r = type_breakdown[t]
    print(f'  {t:6s} (n={r["n_test"]}, k={r["n_known"]}, u={r["n_unknown"]}):')
    print(f'         sae feat   = {r["sae_feature"]["mean"]:.3f}  [{r["sae_feature"]["lo"]:.3f}, {r["sae_feature"]["hi"]:.3f}]')
    print(f'         lin probe  = {r["linear_probe"]["mean"]:.3f}  [{r["linear_probe"]["lo"]:.3f}, {r["linear_probe"]["hi"]:.3f}]')
    print(f'         diff-mean  = {r["diff_means"]["mean"]:.3f}  [{r["diff_means"]["lo"]:.3f}, {r["diff_means"]["hi"]:.3f}]')

## 11. Save artifacts + upload to HF

In [ ]:
from huggingface_hub import HfApi
from datetime import datetime, timezone

per_layer = {
    str(L): {
        'sae_feature': sae_results.get(L),
        'linear_probe': linprobe_results[L],
        'diff_means':   diffmeans_results[L],
    }
    for L in ALL_LAYERS
}

summary = {
    'version':         'paper-baselines-v1',
    'model':           HF_BASE_MODEL,
    'sae_repo':        HF_SAE_REPO,
    'n_total':         int(n_total),
    'n_known':         int(y_label.sum()),
    'n_unknown':       int(n_total - y_label.sum()),
    'train_size':      len(tr_idx),
    'test_size':       len(te_idx),
    'pile_filter':     {'n_pile_tokens': n_pile_tok_total, 'threshold': PILE_FILTER_THRESHOLD},
    'bootstrap_n':     BOOTSTRAP_N,
    'methods':         ['single_sae_latent (L11/L31/L55 only)', 'linear_probe_l2 (all 64 layers)', 'diff_of_means (all 64 layers)'],
    'best_linprobe_layer':  int(best_linprobe_layer),
    'best_linprobe_auroc':  float(linprobe_results[best_linprobe_layer]['auroc_point']),
    'best_diffmeans_layer': int(best_dm_layer),
    'best_diffmeans_auroc': float(diffmeans_results[best_dm_layer]['auroc_point']),
    'sae_at_L11':       sae_results.get(11),
    'sae_at_L31':       sae_results.get(31),
    'sae_at_L55':       sae_results.get(55),
    'per_type_at_L11':  type_breakdown,
    'ferrando_baseline': 0.732,
    'reference':        'Ferrando 2024 ICLR — arxiv:2411.14257',
    'timestamp':        datetime.now(timezone.utc).isoformat(),
}

with open('/content/paper_baselines_per_layer.json', 'w') as f:
    json.dump(per_layer, f, indent=2)
with open('/content/paper_baselines_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
with open('/content/paper_baselines_per_type.json', 'w') as f:
    json.dump(type_breakdown, f, indent=2)

api = HfApi()
for fname in ['paper_baselines_per_layer.json', 'paper_baselines_summary.json',
              'paper_baselines_per_type.json', 'fig_per_layer_comparison.pdf',
              'fig_per_layer_comparison.png']:
    src = f'/content/{fname}'
    if os.path.exists(src):
        api.upload_file(
            path_or_fileobj=src,
            path_in_repo=f'paper_baselines/{fname}',
            repo_id=HF_SAE_REPO,
            commit_message=f'Paper baselines (linprobe + diff-means + SAE × bootstrap CI) — {fname}',
        )
        print(f'  ✓ uploaded {fname}')

print(f'\n✓ all artifacts under https://huggingface.co/{HF_SAE_REPO}/tree/main/paper_baselines')
print(f'\nHEADLINES for the paper:')
print(f'  · SAE feature L11/f{sae_results[11]["best_feature"]}: AUROC = {sae_results[11]["auroc_point"]:.4f}'
      f'  CI=[{sae_results[11]["auroc_ci_lo"]:.3f}, {sae_results[11]["auroc_ci_hi"]:.3f}]')
print(f'  · Linear probe L{best_linprobe_layer} (best): AUROC = {linprobe_results[best_linprobe_layer]["auroc_point"]:.4f}'
      f'  CI=[{linprobe_results[best_linprobe_layer]["auroc_ci_lo"]:.3f}, {linprobe_results[best_linprobe_layer]["auroc_ci_hi"]:.3f}]')
print(f'  · Diff-of-means L{best_dm_layer} (best): AUROC = {diffmeans_results[best_dm_layer]["auroc_point"]:.4f}')
print(f'  · Ferrando 2024 baseline (Gemma-2-2B-IT, L13): AUROC = 0.732')